# 🔍 Agentic RAG with Azure AI Foundry (Python)

## 📋 Learning Objectives

This notebook demonstrates how to build intelligent Retrieval-Augmented Generation (RAG) systems using the Microsoft Agent Framework with Azure AI Foundry. You'll learn to create agents that can search through documents and provide accurate, grounded responses based on uploaded content.

**Key RAG Capabilities You'll Build:**
- 📚 **Document Upload**: Upload and process documents for agent access
- 🔍 **Vector Store Creation**: Create searchable vector stores for semantic retrieval
- 🤖 **Grounded Responses**: Build agents that only answer from document content
- 🛡️ **Source Attribution**: Ensure responses are traceable to source documents

## 🎯 RAG Architecture Concepts

### Core RAG Components
- **Document Processing**: Upload files to Azure AI Foundry for indexing
- **Vector Store**: Semantic search capability for relevant content retrieval
- **File Search Tool**: Agent tool that queries the vector store
- **Grounded Generation**: AI responses based only on retrieved context

### Technical Flow
```python
Document Upload → Vector Store Creation → Agent Configuration
                        ↓                       ↓
User Query → File Search Tool → Retrieved Context → Grounded Response
```

## 🏗️ Technical Architecture

### Azure AI Foundry Integration
- **FoundryChatClient**: Foundry-native client for the Responses API
- **Vector Store Management**: Handles document indexing and retrieval via `client.client`
- **File Search Tool**: Built-in tool via `client.get_file_search_tool()`

### Agent Behavior
- Only answers questions using information from uploaded documents
- Clearly indicates when questions cannot be answered from available content
- Provides explicit citations to source documents
- Avoids general knowledge responses outside document scope

## ⚙️ Prerequisites & Setup

**Required Dependencies:**
```bash
pip install agent-framework-foundry azure-identity -U
```

**Azure Configuration:**
- Azure subscription with AI Foundry access
- Azure CLI authentication configured
- Environment variables in `.env` file:
  ```env
  FOUNDRY_PROJECT_ENDPOINT=your_project_endpoint
  FOUNDRY_MODEL=your_model_deployment
  ```

**Authentication:**
```bash
az login --tenant <Your Azure Tenant>
```

## 📊 Use Cases

### Knowledge Management
- Corporate document Q&A systems
- Policy and procedure guidance
- Technical documentation assistants
- Training material search

### Customer Support
- Product documentation queries
- Troubleshooting guide search
- FAQ and knowledge base systems
- Support ticket assistance

## 🚀 Getting Started

Follow the cells below to:
1. Install required dependencies and import libraries
2. Create a vector store with sample documents
3. Configure an agent with file search capabilities
4. Query the agent and receive grounded responses

Let's build an intelligent document-aware assistant! 📖✨

In [1]:
# ✅ Dependencies installed via terminal:
# uv venv .venv --python 3.13
# uv pip install ipykernel --python .venv/Scripts/python.exe
# uv pip install -r Installation/requirements.txt --python .venv/Scripts/python.exe
#
# Select the .venv kernel in VS Code to use these packages.

In [2]:
import os
import contextlib

from azure.identity import AzureCliCredential
from dotenv import load_dotenv

from agent_framework import Agent
from agent_framework.foundry import FoundryChatClient

## 📦 Import Required Libraries

Import FoundryChatClient for Foundry Responses API integration and Agent for agent creation.

In [3]:
from dotenv import find_dotenv
load_dotenv(find_dotenv())

True

## 🔧 Load Environment Variables

Load Azure AI Foundry configuration from the `.env` file.

In [4]:
# Define the query for the agent
query = "Can you explain Contoso's travel insurance coverage?"

## ❓ Define Query

Set up the question to ask the RAG agent about the uploaded document.

In [5]:
client = FoundryChatClient(credential=AzureCliCredential())

file = None
vector_store = None

try:
    # 1. Upload file and create vector store
    file_path = './document.md'
    print(f"Uploading file from: {file_path}")

    file = await client.client.files.create(
        file=open(file_path, "rb"), purpose="assistants"
    )
    print(f"Uploaded file, file ID: {file.id}")

    vector_store = await client.client.vector_stores.create(
        name="contoso_knowledge_base",
        expires_after={"anchor": "last_active_at", "days": 1},
    )
    await client.client.vector_stores.files.create_and_poll(
        vector_store_id=vector_store.id, file_id=file.id
    )
    print(f"Created vector store, vector store ID: {vector_store.id}")

    # 2. Create file search tool
    file_search_tool = client.get_file_search_tool(vector_store_ids=[vector_store.id])

    # 3. Create an agent with file search capabilities
    agent = Agent(
        client=client,
        name="PythonRAGAgent",
        instructions=(
            "You are an AI assistant designed to answer user questions using only the information "
            "retrieved from the provided document(s). "
            "If a user's question cannot be answered using the retrieved context, you must clearly respond: "
            "'I'm sorry, but the uploaded document does not contain the necessary information to answer that question.' "
            "Do not answer from general knowledge or reasoning. Do not make assumptions or generate hypothetical explanations. "
            "For questions that do have relevant content in the document, respond accurately and cite the document explicitly."
        ),
        tools=[file_search_tool],
    )

    # 4. Query the agent
    print(f"\n# User: '{query}'")
    response = await agent.run(query)
    print(f"# Agent: {response.text}")

finally:
    # 5. Cleanup: Delete the vector store and file
    if vector_store:
        with contextlib.suppress(Exception):
            await client.client.vector_stores.delete(vector_store_id=vector_store.id)
            print(f"\nDeleted vector store: {vector_store.id}")
    if file:
        with contextlib.suppress(Exception):
            await client.client.files.delete(file_id=file.id)
            print(f"Deleted file: {file.id}")

C:\Users\nstijepovic\AppData\Local\Temp\ipykernel_29500\2239225031.py:7: DeprecationWarning: AzureAIProjectAgentProvider is deprecated. Use FoundryAgent instead.
  AzureAIProjectAgentProvider(credential=credential) as provider,


Uploading file from: ./document.md


ServiceRequestError: Bearer token authentication is not permitted for non-TLS protected (non-https) URLs.

## 🚀 Create and Run the RAG Agent

This cell performs the complete RAG workflow:

**Workflow Steps:**
1. **Upload Document**: Upload the document file to Azure AI Foundry
2. **Create Vector Store**: Build a searchable vector index from the document
3. **Configure File Search Tool**: Set up the tool with `client.get_file_search_tool()`
4. **Create Agent**: Use `Agent` with `FoundryChatClient` for agent creation
5. **Query Agent**: Run the query and get grounded responses
6. **Cleanup Resources**: Automatically delete vector store and file to prevent resource leaks